# Cleaning Stage - Pokemon Legendary Classifier

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import seaborn as sns

### Take a look to our dataset

We began by loading the Pokémon dataset from our CSV file to perform exploratory data analysis, allowing us to identify the key features and metrics available for training our Legendary Classifier model

In [3]:
# Reading our dataset
df_pokemon = pd.read_csv(r"..\..\Data_Scientist_Training\Pokemon-Legendary-Classifier\datasets\pokemon.csv")

In [4]:
# Checking out our dataset
df_pokemon.head(3)

,abilities,against_bug,against_dark,against_dragon,against_electric,against_fairy,against_fight,against_fire,against_flying,against_ghost,...,percentage_male,pokedex_number,sp_attack,sp_defense,speed,type1,type2,weight_kg,generation,is_legendary
0,"['Overgrow', 'Chlorophyll']",1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,...,88.1,1,65,65,45,grass,poison,6.9,1,0
1,"['Overgrow', 'Chlorophyll']",1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,...,88.1,2,80,80,60,grass,poison,13.0,1,0
2,"['Overgrow', 'Chlorophyll']",1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,...,88.1,3,122,120,80,grass,poison,100.0,1,0


In [5]:
# Checking out columns 
df_pokemon.columns

Index(['abilities', 'against_bug', 'against_dark', 'against_dragon',
       'against_electric', 'against_fairy', 'against_fight', 'against_fire',
       'against_flying', 'against_ghost', 'against_grass', 'against_ground',
       'against_ice', 'against_normal', 'against_poison', 'against_psychic',
       'against_rock', 'against_steel', 'against_water', 'attack',
       'base_egg_steps', 'base_happiness', 'base_total', 'capture_rate',
       'classfication', 'defense', 'experience_growth', 'height_m', 'hp',
       'japanese_name', 'name', 'percentage_male', 'pokedex_number',
       'sp_attack', 'sp_defense', 'speed', 'type1', 'type2', 'weight_kg',
       'generation', 'is_legendary'],
      dtype='str')

### Choosing our metrics and getting an overall overview

An inspection of the columns revealed key performance statistics such as Attack, Defense, and Speed, which will serve as the primary features for model training. Before fitting the model, we will preprocess the data to handle inconsistencies and outliers, and engineer a new feature representing the cumulative sum of all base stats.

Besides that, as our dataset is big and has some columns that we'll not use. Will drop them to decrease the weight of our data and avoid noise

In [10]:
# Drop unnecessary columns to avoid noise and decrease our dataset weight
columns_to_drop = [
    'abilities',
    'against_bug',
    'against_dark', 
    'against_dragon',
    'against_electric', 
    'against_fairy',
    'against_fight', 
    'against_fire',
    'against_flying', 
    'against_ghost', 
    'against_grass', 
    'against_ground',
    'against_ice', 
    'against_normal', 
    'against_poison', 
    'against_psychic',
    'against_rock', 
    'against_steel', 
    'against_water',
    'base_egg_steps', 
    'base_happiness', 
    'base_total', 
    'capture_rate',
    'classfication',
    'experience_growth',
    'height_m',
    'weight_kg',
    'generation', 
    'is_legendary',
    'japanese_name',
    'percentage_male'
]

df_pokemon = df_pokemon.drop(columns = columns_to_drop)

In [11]:
# Make sure unncessary columns where deleted
df_pokemon.columns

Index(['attack', 'defense', 'hp', 'name', 'pokedex_number', 'sp_attack',
       'sp_defense', 'speed', 'type1', 'type2'],
      dtype='str')

#### Checking out more information and statistics from our dataset

In [12]:
# Getting more information about each column
df_pokemon.info()

<class 'pandas.DataFrame'>
RangeIndex: 801 entries, 0 to 800
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   attack          801 non-null    int64
 1   defense         801 non-null    int64
 2   hp              801 non-null    int64
 3   name            801 non-null    str  
 4   pokedex_number  801 non-null    int64
 5   sp_attack       801 non-null    int64
 6   sp_defense      801 non-null    int64
 7   speed           801 non-null    int64
 8   type1           801 non-null    str  
 9   type2           417 non-null    str  
dtypes: int64(7), str(3)
memory usage: 62.7 KB


As shown above, the dataset contains no missing values, with the sole exception of the `Type2` column. This is expected, as many Pokémon are single-type and do not possess a secondary attribute.

In [13]:
# Getting a statistical overview of our dataset
df_pokemon.describe()

,attack,defense,hp,pokedex_number,sp_attack,sp_defense,speed
count,801.000000,801.000000,801.000000,801.000000,801.000000,801.000000,801.000000
mean,77.857678,73.008739,68.958801,401.000000,71.305868,70.911361,66.334582
std,32.158820,30.769159,26.576015,231.373075,32.353826,27.942501,28.907662
min,5.000000,5.000000,1.000000,1.000000,10.000000,20.000000,5.000000
25%,55.000000,50.000000,50.000000,201.000000,45.000000,50.000000,45.000000
50%,75.000000,70.000000,65.000000,401.000000,65.000000,66.000000,65.000000
75%,100.000000,90.000000,80.000000,601.000000,91.000000,90.000000,85.000000
max,185.000000,230.000000,255.000000,801.000000,194.000000,230.000000,180.000000


Most of them make sense, with the exception of the minimum values for certain stats like attack and defense, which likely correspond to first-stage evolution Pokémon.

### Start Cleaning Stage

#### Dealing with missing values

In [ ]:
# Summing null values for each column to see how many we have
df_pokemon.isna().sum()

attack              0
defense             0
hp                  0
name                0
pokedex_number      0
sp_attack           0
sp_defense          0
speed               0
type1               0
type2             384
dtype: int64

In [16]:
# To make sure are pokemons with a single type, lets watch the rows
df_pokemon[df_pokemon.isna().any(axis = 1)]

,attack,defense,hp,name,pokedex_number,sp_attack,sp_defense,speed,type1,type2
3,52,43,39,Charmander,4,60,50,65,fire,NaN
4,64,58,58,Charmeleon,5,80,65,80,fire,NaN
6,48,65,44,Squirtle,7,50,64,43,water,NaN
7,63,80,59,Wartortle,8,65,80,58,water,NaN
8,103,120,79,Blastoise,9,135,115,78,water,NaN
...,...,...,...,...,...,...,...,...,...,...
781,55,65,45,Jangmo-o,782,45,45,45,dragon,NaN
788,29,31,43,Cosmog,789,29,31,37,psychic,NaN
789,29,131,43,Cosmoem,790,29,131,37,psychic,NaN
795,89,71,83,Xurkitree,796,173,71,83,electric,NaN


We confirm our hypothesis we mentioned previously, these are pokemon with a single type

In [18]:
'''
Some of these pokemons have good stats for our Legendary Classifier, so we'll fill the null values with
None to not drop all of these rows
'''

df_pokemon['type2'] = df_pokemon['type2'].fillna(value = 'None')

In [19]:
# Make sure null values where deleted
df_pokemon.isna().sum()

attack            0
defense           0
hp                0
name              0
pokedex_number    0
sp_attack         0
sp_defense        0
speed             0
type1             0
type2             0
dtype: int64

#### Looking for inconsistent typos and text

Here we are making sure that each pokemon type don't have typos or any missing text, after a caregul review we can see that everything is in order and nothing out of context has been found

In [ ]:
# Checking out for inconsistent typos and text for type 1
df_pokemon['type1'].value_counts()

type1
water       114
normal      105
grass        78
bug          72
psychic      53
fire         52
rock         45
electric     39
poison       32
ground       32
dark         29
fighting     28
ghost        27
dragon       27
steel        24
ice          23
fairy        18
flying        3
Name: count, dtype: int64

In [21]:
# Checking out for inconsistent typos and text for type 1
df_pokemon['type2'].value_counts()

type2
None        384
flying       95
poison       34
ground       34
fairy        29
psychic      29
fighting     25
steel        22
dark         21
grass        20
water        17
dragon       17
ice          15
rock         14
ghost        14
fire         13
electric      9
bug           5
normal        4
Name: count, dtype: int64

#### Checking for duplicate values

Giving a quick check for duplicate values where fortunate there are no presence of it

In [23]:
# Checking out for duplicate rows
df_pokemon[df_pokemon.duplicated(keep = False)]

,attack,defense,hp,name,pokedex_number,sp_attack,sp_defense,speed,type1,type2


#### Changing index number to pokedex number

Since each Pokémon has a unique Pokédex number, we will set this column as the index of our DataFrame.

In [24]:
df_pokemon.head(3)

,attack,defense,hp,name,pokedex_number,sp_attack,sp_defense,speed,type1,type2
0,49,49,45,Bulbasaur,1,65,65,45,grass,poison
1,62,63,60,Ivysaur,2,80,80,60,grass,poison
2,100,123,80,Venusaur,3,122,120,80,grass,poison


In [26]:
# Setting pokedex number to be the index of our dataset
df_pokemon.set_index('pokedex_number', inplace = True)

In [31]:
df_pokemon.head(3)

,attack,defense,hp,name,sp_attack,sp_defense,speed,type1,type2
pokedex_number,,,,,,,,,
1,49,49,45,Bulbasaur,65,65,45,grass,poison
2,62,63,60,Ivysaur,80,80,60,grass,poison
3,100,123,80,Venusaur,122,120,80,grass,poison
